In [8]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("argilla/ultrafeedback-binarized-preferences-cleaned")

In [9]:
df = pd.DataFrame(dataset)

df.head()

,train
0,"{'source': 'evol_instruct', 'prompt': 'Can you..."
1,"{'source': 'evol_instruct', 'prompt': 'Suppose..."
2,"{'source': 'evol_instruct', 'prompt': 'Identif..."
3,"{'source': 'evol_instruct', 'prompt': 'How can..."
4,"{'source': 'evol_instruct', 'prompt': 'Can you..."


In [10]:
import json
# select the first row (change index if you want a different row)
row = df.iloc[0]
print(json.dumps(row.to_dict(), indent=2, ensure_ascii=False))

{
  "train": {
    "source": "evol_instruct",
    "prompt": "Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here's some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << \"Enter the name of a country: \";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}",
    "chosen": [
      {
        "content": "Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here's some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << \"Enter the name of a country: \";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++

In [11]:
import jsonlines
from pathlib import Path

In [14]:
# Write to JSON lines file
output_folder = Path(Path.cwd() / "content")
output_folder.mkdir(exist_ok=True)

# Use JSON Lines (.jsonl) so each item is a separate JSON object on its own line
file_path = output_folder / 'QA_prompts_and_completions.jsonl'
# Write using the jsonlines library for correct JSONL formatting
items = []
for idx, row in df.iterrows():
    row_dic = row.iloc[0]
    row_dic["index"] = idx
    items.append(row_dic)

with jsonlines.open(str(file_path), mode='w') as writer:
    writer.write_all(items)

## Download model

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"

model = AutoModelForCausalLM.from_pretrained(model_name, load_in_4bit=True, device_map="auto") #lower precision to finetune
tokenzier = AutoTokenizer.from_pretrained(model_name)


d:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 3/3 [00:07<00:00,  2.36s/it]


In [2]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear4bit(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
 

In [4]:
#QLoRA for faster finetuning
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16, #rank of matrices
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'], #other layers are frozen
    lora_dropout=0.05,
    bias='none',
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

d:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(


In [15]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [16]:
import torch.nn.functional as F
from torch.utils.data import DataLoader

In [17]:
ref_model = model #frozen model
ref_model.eval()

train_model = model
train_model.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Lin

In [18]:
def _util_fn(batch, tokenizer, max_length = 1024):
    prompts = [x['prompt'] for x in batch]
    chosen = [x['chosen'] for x in batch]
    rejected = [x['rejected'] for x in batch]

    chosen_texts = [p + c for p, c in zip(prompts, chosen)]
    rejected_texts = [p + c for p, c in zip(prompts, rejected)]

    chosen_enc = tokenizer(
        chosen_texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    rejected_enc = tokenizer(
        rejected_texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    prompt_lens = [ #useful for masking
        len(tokenizer(p)) for p in prompts
    ]

    return {
        "chosen_input_ids": chosen_enc["input_ids"],
        "chosen_attention_mask": chosen_enc["attention_mask"],
        "rejected_input_ids": rejected_enc["input_ids"],
        "rejected_attention_mask": rejected_enc["attention_mask"],
        "prompt_lens": torch.tensor(prompt_lens)
    }

In [20]:
def compute_logps(model, input_ids, attention_mask, prompt_lens):
    outputs = model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits

    log_probs = F.log_softmax(logits, dim=-1)

    shift_logits = log_probs[:, :-1, :]
    shift_labels = input_ids[:, 1:]

    selected = shift_logits.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1) #selected[i, t] = logprob(t+1 | tokens<=t)

    mask = torch.ones_like(shift_labels) #general mask

    for idx, pl in enumerate(prompt_lens):
        mask[idx, :pl-1] = 0
    
    selected = selected * mask #maintain only logprobs of otkens not in prompt

    #for each sentence, return the total logprob (summing in last dimension)
    return selected.sum(dim=-1)

In [ ]:
def dpo_loss(motrain_model, ref_model, batch, beta=0.1):
    